# 🏭 AIoT Multi-Sensor Anomaly Detection Workshop (Colab Edition)
### เปรียบเทียบโมเดล Isolation Forest vs One-Class SVM บนข้อมูลบอร์ด Cucumber RIS (HTS221 + BMP280 + MPU6050)
**วิชา:** CN466 Project - Internet of Things & AI | **ผู้พัฒนา:** ชนาภา สดุดี (6610680214)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
print('Libraries ready!')

## 1. สร้างข้อมูล Multi-Sensor Telemetry (ปกติ + คอมเพรสเซอร์ดับ + แรงกระแทก MPU6050)

In [ ]:
np.random.seed(42)
n = 600
ts = pd.date_range('2026-11-20 08:00:00', periods=n, freq='2s')
temp = 4.2 + np.random.normal(0, 0.14, n)
humid = 60.0 + np.random.normal(0, 0.8, n)
pressure = 1012.5 + np.random.normal(0, 0.15, n)
vib_g = np.abs(np.random.normal(0.02, 0.01, n))
y_true = np.zeros(n, dtype=int)

# เหตุการณ์ที่ 1: แรงกระแทก/เปิดประตูแรง (step 180-195)
vib_g[180:195] = np.random.uniform(0.55, 1.25, 15)
pressure[180:195] -= np.random.uniform(1.5, 3.0, 15)
y_true[180:195] = 1

# เหตุการณ์ที่ 2: คอมเพรสเซอร์ดับ (step 420-600)
for i in range(420, n):
    temp[i] = temp[i-1] + np.random.uniform(0.06, 0.14)
    humid[i] = humid[i-1] + np.random.uniform(0.08, 0.2)
y_true[420:] = 1

df = pd.DataFrame({'ts': ts, 'temp': temp, 'humid': humid, 'pressure': pressure, 'vib_g': vib_g, 'label': y_true})
df['temp_slope'] = df['temp'].diff(5).fillna(0) / 5.0
df['z_score'] = (df['temp'] - df['temp'].rolling(15).mean().bfill()) / (df['temp'].rolling(15).std().bfill() + 1e-4)
df.head()

## 2. เปรียบเทียบโมเดล Isolation Forest vs One-Class SVM

In [ ]:
cols = ['temp', 'humid', 'pressure', 'vib_g', 'temp_slope', 'z_score']
X = StandardScaler().fit_transform(df[cols])
X_train_normal = X[:150]

iso = IsolationForest(contamination=0.05, random_state=42).fit(X_train_normal)
svm = OneClassSVM(nu=0.05, kernel='rbf', gamma='scale').fit(X_train_normal)

df['pred_iso'] = (iso.predict(X) == -1).astype(int)
df['pred_svm'] = (svm.predict(X) == -1).astype(int)

print('=== Isolation Forest ===')
print(classification_report(df['label'], df['pred_iso'], target_names=['Normal', 'Anomaly']))
print('=== One-Class SVM ===')
print(classification_report(df['label'], df['pred_svm'], target_names=['Normal', 'Anomaly']))

## 3. แสดงผลกราฟการตรวจจับ Multi-Sensor Anomalies

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(df['ts'], df['temp'], label='Temp (C)', color='#d97706')
ax1.axhline(8.0, color='#8c181c', linestyle='--', label='GMP Max (8C)')
anom = df[df['pred_iso'] == 1]
ax1.scatter(anom['ts'], anom['temp'], color='#dc2626', s=18, label='Isolation Forest Alert')
ax1.set_ylabel('Temp (C)'); ax1.legend(); ax1.grid(alpha=0.25)

ax2.plot(df['ts'], df['vib_g'], label='MPU6050 Vibration (G)', color='#2563eb')
ax2.axhline(0.35, color='red', linestyle='--', label='Shock Threshold (0.35g)')
ax2.set_ylabel('Vibration (G)'); ax2.legend(); ax2.grid(alpha=0.25)
plt.suptitle('Multi-Sensor Cold Chain Anomaly Detection (HTS221 + BMP280 + MPU6050)')
plt.tight_layout(); plt.show()